### Data Loading

In [1]:
import pandas as pd

load_df = pd.read_csv("data/force/f_train.csv")
drop_columns = ['RSHA', 'RMED', 'GR', 'PEF', 'SP', 'ROP', 'DRHO', 'RDEP', 'WELL']
load_df = load_df.drop(drop_columns, axis=1)
load_df = load_df.dropna()
df = load_df.copy()

# Categorize Lithology
litho_code = {30000: "Sandstone", 65030: "SandstoneShale", 65000: "Shale", 80000: "Marl", 
            74000: "Dolomite", 70000: "Limestone", 70032: "Chalk", 88000: "Halite", 
            86000: "Anhydrite", 99000: "Tuff", 90000: "Coal", 93000: "Basement"}
df["LITHOLOGY"] = df["LITHOLOGY"].map(litho_code)

df = df[df.LITHOLOGY != "Shale"]
df = df.reset_index(drop=True)

drop = ['DEPTH_MD', 'GROUP', 'FORMATION', 'BS']
dt_df = df.drop(drop, axis=1)

# Select sample that Litho is in ['Limestone', 'SandstoneShale']
dt_df = dt_df[dt_df["LITHOLOGY"].isin(["Limestone", "SandstoneShale"])]
dt_df = dt_df.reset_index(drop=True)
dt_df.head()

,CALI,RHOB,NPHI,DTC,LITHOLOGY
0,17.363323,2.000440,0.589035,133.393707,SandstoneShale
1,17.306395,2.007255,0.595443,135.019684,SandstoneShale
2,17.303955,2.017085,0.610448,136.660767,SandstoneShale
3,17.368221,2.027622,0.614587,136.615814,SandstoneShale
4,17.433296,2.033576,0.600994,134.936020,SandstoneShale


#### Convert Label to Binary values

In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
dt_df['LITHOLOGY'] = le.fit_transform(dt_df['LITHOLOGY'])
dt_df.LITHOLOGY.value_counts()

LITHOLOGY
0    2523
1    2386
Name: count, dtype: int64

In [3]:
dt_df.head()

,CALI,RHOB,NPHI,DTC,LITHOLOGY
0,17.363323,2.000440,0.589035,133.393707,1
1,17.306395,2.007255,0.595443,135.019684,1
2,17.303955,2.017085,0.610448,136.660767,1
3,17.368221,2.027622,0.614587,136.615814,1
4,17.433296,2.033576,0.600994,134.936020,1


#### Max-Min Normalization

In [4]:
# Normalize the data
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(dt_df.drop('LITHOLOGY', axis=1))
scaled_df = pd.DataFrame(scaled_features, columns=dt_df.columns[:-1])
scaled_df['LITHOLOGY'] = dt_df['LITHOLOGY'].values
scaled_df.head()


,CALI,RHOB,NPHI,DTC,LITHOLOGY
0,0.609454,0.417278,0.724849,0.697913,1
1,0.605952,0.421766,0.733094,0.710956,1
2,0.605802,0.428240,0.752399,0.724119,1
3,0.609755,0.435180,0.757723,0.723759,1
4,0.613757,0.439101,0.740235,0.710284,1


### Train-Test Splitting

In [5]:
# Train-test split
from sklearn.model_selection import train_test_split
X = scaled_df.drop('LITHOLOGY', axis=1)
y = scaled_df['LITHOLOGY']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Logistic Regression

In [6]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(max_iter=500, tol=0.001, random_state=42)
log_reg.fit(X_train, y_train)

LogisticRegression(max_iter=500, random_state=42, tol=0.001)

#### Evaluate the model

In [7]:
# Evaluate the model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
y_pred = log_reg.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Accuracy: {accuracy:.4f}")

Logistic Regression Accuracy: 0.8768


#### Confusion Matrix

In [8]:
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.86      0.88       505
           1       0.86      0.90      0.88       477

    accuracy                           0.88       982
   macro avg       0.88      0.88      0.88       982
weighted avg       0.88      0.88      0.88       982

Confusion Matrix:
 [[433  72]
 [ 49 428]]
